# Desafio Extra — Introdução ao Data Science

**Projeto:** Análise Exploratória de Dados — Sample Superstore  
**Autor:** Daniel Quiteque  
**Dataset:** Sample Superstore

## Objetivo
Realizar uma Análise Exploratória de Dados (AED) sobre uma rede varejista, compreendendo vendas, lucro, descontos, categorias, segmentos, regiões e tendências temporais. O foco é transformar dados brutos em informações úteis para apoiar decisões de negócio.


## 1. Importação e compreensão dos dados

Nesta etapa, o dataset é carregado e são verificadas dimensões, tipos de dados, valores ausentes, duplicados e estatísticas descritivas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_raw = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

print("Dimensão:", df_raw.shape)
display(df_raw.head())
display(df_raw.dtypes.to_frame("tipo"))
print("\nValores nulos:", int(df_raw.isna().sum().sum()))
print("Duplicados:", int(df_raw.duplicated().sum()))
display(df_raw.describe(include="all").T)


## 2. Tratamento e preparação

Foram padronizados os nomes das colunas e convertidas `Order Date` e `Ship Date` para o tipo `datetime`. O dataset analisado não possui valores nulos nem linhas totalmente duplicadas.

Os outliers foram identificados pelo método do intervalo interquartil (IQR). Em variáveis financeiras, valores extremos podem representar compras reais de grande porte; por isso, eles não foram eliminados dos cálculos de vendas e lucro. Para evitar distorção visual, foram criadas versões winsorizadas de `sales` e `profit` apenas para gráficos de dispersão. Assim, preservam-se os resultados reais e melhora-se a leitura das visualizações.


In [ ]:
df = df_raw.copy()

df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_", regex=False)
              .str.replace("-", "_", regex=False)
)

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")

def detectar_outliers_iqr(data, coluna):
    q1 = data[coluna].quantile(0.25)
    q3 = data[coluna].quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    mask = (data[coluna] < limite_inf) | (data[coluna] > limite_sup)
    return {
        "variavel": coluna,
        "limite_inferior": limite_inf,
        "limite_superior": limite_sup,
        "qtd_outliers": int(mask.sum())
    }

relatorio_outliers = pd.DataFrame([
    detectar_outliers_iqr(df, c)
    for c in ["sales", "profit", "quantity", "discount"]
])

display(relatorio_outliers)

df["sales_viz"] = df["sales"].clip(
    df["sales"].quantile(0.01),
    df["sales"].quantile(0.99)
)
df["profit_viz"] = df["profit"].clip(
    df["profit"].quantile(0.01),
    df["profit"].quantile(0.99)
)

df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["profit_margin"] = np.where(df["sales"] != 0, df["profit"] / df["sales"], np.nan)
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days


## 3. Indicadores gerais (KPIs)

Os primeiros indicadores permitem compreender a dimensão da operação: vendas totais, lucro total, margem, quantidade de pedidos, clientes e ticket médio.


In [ ]:
vendas_totais = df["sales"].sum()
lucro_total = df["profit"].sum()
margem_total = lucro_total / vendas_totais
pedidos = df["order_id"].nunique()
clientes = df["customer_id"].nunique()
ticket_medio = df.groupby("order_id")["sales"].sum().mean()

print(f"Vendas totais: US$ {vendas_totais:,.2f}")
print(f"Lucro total: US$ {lucro_total:,.2f}")
print(f"Margem de lucro: {margem_total:.2%}")
print(f"Pedidos únicos: {pedidos:,}")
print(f"Clientes únicos: {clientes:,}")
print(f"Ticket médio por pedido: US$ {ticket_medio:,.2f}")


## 4. Vendas e lucro por categoria

O `GroupBy` é usado para comparar as três categorias principais e identificar quais combinam volume de vendas e rentabilidade.


In [ ]:
categoria = (
    df.groupby("category")
      .agg(vendas=("sales", "sum"),
           lucro=("profit", "sum"),
           quantidade=("quantity", "sum"))
      .sort_values("vendas", ascending=False)
)

categoria["margem_pct"] = categoria["lucro"] / categoria["vendas"] * 100
display(categoria)

fig, ax = plt.subplots(figsize=(8,5))
categoria["vendas"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Vendas por categoria")
ax.set_xlabel("Vendas (US$)")
plt.tight_layout()
plt.show()


## 5. Subcategorias e rentabilidade

Analisar somente vendas pode esconder problemas de rentabilidade. Por isso, também são avaliadas subcategorias com lucro negativo.


In [ ]:
subcategoria = (
    df.groupby("sub_category")
      .agg(vendas=("sales", "sum"),
           lucro=("profit", "sum"),
           quantidade=("quantity", "sum"))
      .sort_values("lucro")
)

display(subcategoria)

plt.figure(figsize=(10,7))
subcategoria["lucro"].plot(kind="barh")
plt.axvline(0, linewidth=1)
plt.title("Lucro por subcategoria")
plt.xlabel("Lucro (US$)")
plt.tight_layout()
plt.show()


## 6. Impacto dos descontos na rentabilidade

A relação entre `discount` e `profit` é investigada por agrupamento e por gráfico de dispersão. Isso permite verificar se níveis maiores de desconto estão associados a menor rentabilidade.


In [ ]:
desconto = (
    df.groupby("discount")
      .agg(transacoes=("row_id", "count"),
           vendas=("sales", "sum"),
           lucro=("profit", "sum"),
           lucro_medio=("profit", "mean"))
      .reset_index()
)

display(desconto)

correlacao = df[["discount", "profit"]].corr().iloc[0,1]
print(f"Correlação desconto x lucro: {correlacao:.3f}")

plt.figure(figsize=(8,5))
plt.scatter(df["discount"], df["profit_viz"], alpha=0.25)
plt.axhline(0, linewidth=1)
plt.title("Desconto x lucro")
plt.xlabel("Desconto")
plt.ylabel("Lucro (winsorizado para visualização)")
plt.tight_layout()
plt.show()


## 7. Desempenho por segmento e região

Esta etapa aplica filtros, ordenações e agrupamentos para compreender quais perfis de clientes e regiões apresentam maior contribuição para vendas e lucro.


In [ ]:
segmento = (
    df.groupby("segment")
      .agg(vendas=("sales", "sum"),
           lucro=("profit", "sum"),
           pedidos=("order_id", "nunique"))
      .sort_values("vendas", ascending=False)
)

regiao = (
    df.groupby("region")
      .agg(vendas=("sales", "sum"),
           lucro=("profit", "sum"),
           pedidos=("order_id", "nunique"))
      .sort_values("vendas", ascending=False)
)

display(segmento)
display(regiao)

print("\nCinco estados mais lucrativos:")
display(
    df.groupby("state")
      .agg(vendas=("sales", "sum"), lucro=("profit", "sum"))
      .sort_values("lucro", ascending=False)
      .head()
)

print("\nCinco estados com menor lucro:")
display(
    df.groupby("state")
      .agg(vendas=("sales", "sum"), lucro=("profit", "sum"))
      .sort_values("lucro")
      .head()
)


## 8. Tendência temporal

A evolução anual e mensal mostra como vendas e lucro mudaram no período da base, permitindo observar crescimento e sazonalidade.


In [ ]:
anual = (
    df.groupby("year")
      .agg(vendas=("sales", "sum"),
           lucro=("profit", "sum"),
           pedidos=("order_id", "nunique"))
)
display(anual)

mensal = (
    df.groupby("month")
      .agg(vendas=("sales", "sum"), lucro=("profit", "sum"))
      .reset_index()
)

plt.figure(figsize=(11,5))
plt.plot(mensal["month"], mensal["vendas"], marker="o", markersize=3)
plt.title("Evolução mensal das vendas")
plt.xlabel("Mês")
plt.ylabel("Vendas (US$)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 9. Principais insights

- **Technology** apresenta o maior volume de vendas e também o maior lucro entre as categorias.
- **Furniture** gera vendas elevadas, mas possui rentabilidade muito inferior, indicando necessidade de atenção à composição da categoria.
- As subcategorias **Tables** e **Bookcases** apresentam lucro acumulado negativo.
- Existe associação negativa entre desconto e lucro: descontos maiores tendem a reduzir a rentabilidade.
- O segmento **Consumer** concentra o maior volume de vendas e lucro.
- A região **West** lidera em vendas e lucro.
- A série anual mostra crescimento relevante no fim do período, com 2017 apresentando o maior volume de vendas.
- A análise de estados revela diferenças importantes de rentabilidade, indicando que decisões comerciais devem considerar localização e mix de produtos.

Esses resultados mostram que crescimento de vendas não deve ser analisado isoladamente. A gestão de descontos e a rentabilidade por categoria e subcategoria são fundamentais para melhorar o desempenho comercial.


## 10. Conclusão

A análise exploratória permitiu transformar o dataset Sample Superstore em informações úteis para tomada de decisão. O projeto percorreu importação, compreensão, padronização, conversão de tipos, identificação de outliers, criação de variáveis derivadas, filtros, ordenações, `GroupBy`, visualizações e interpretação de resultados.

O principal ponto de atenção é a rentabilidade: determinadas subcategorias e níveis de desconto geram prejuízo apesar de existirem vendas. Como próximos passos, seria possível construir um dashboard interativo no Looker Studio, incluir metas comerciais ou desenvolver modelos preditivos de vendas e lucro.
